<a href="https://colab.research.google.com/github/npeyton/ml_materials/blob/main/nielsen_neural_networks_chap_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#### Libraries
# Standard library
import random

# Third-party libraries
import numpy as np

In [ ]:
class Network(object):

    def __init__(self, sizes):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer (hidden layer) 3 neurons,
        and the third layer 1 neuron.  The biases and weights for the
        network are initialized randomly, using a Gaussian
        distribution with mean 0, and variance 1 ( random distribution of numbers from 0-1)."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x)
                        for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)
        return a


  # function for the stochastic gradient descent where you are taking steps to
  # minimize the cost function - with the steps being mini batches.
    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None):
        """Train the neural network using mini-batch stochastic
        gradient descent.  The ``training_data`` is a list of tuples
        ``(x, y)`` representing the training inputs and the desired
        outputs.  The other non-optional parameters are
        self-explanatory.  If ``test_data`` is provided then the
        network will be evaluated against the test data after each
        epoch, and partial progress printed out.  This is useful for
        tracking progress, but slows things down substantially."""
        if test_data: n_test = len(test_data)
        n = len(training_data)
        for j in xrange(epochs):
            random.shuffle(training_data)
            mini_batches = [
                training_data[k:k+mini_batch_size]
                for k in xrange(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            if test_data:
                print ("Epoch {0}: {1} / {2}").format(
                    j, self.evaluate(test_data), n_test)
            else:
                print ("Epoch {0} complete").format(j)

  # Update the weights ab biases after each mini_batch during training.
    def update_mini_batch(self, mini_batch, eta):

        """Update the network's weights and biases by applying
        gradient descent using backpropagation to a single mini batch.
        The ``mini_batch`` is a list of tuples ``(x, y)``, and ``eta``
        is the learning rate."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb+dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw+dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w-(eta/len(mini_batch))*nw
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b-(eta/len(mini_batch))*nb
                      for b, nb in zip(self.biases, nabla_b)]
      # Back propagation function.
    def backprop(self, x, y):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the cost function C_x.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of numpy arrays, similar
        to ``self.biases`` and ``self.weights``."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        # feedforward
        activation = x
        activations = [x] # list to store all the activations, layer by layer
        zs = [] # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation)+b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        # backward pass
        delta = self.cost_derivative(activations[-1], y) * \
            sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        # Note that the variable l in the loop below is used a little
        # differently to the notation in Chapter 2 of the book.  Here,
        # l = 1 means the last layer of neurons, l = 2 is the
        # second-last layer, and so on.  It's a renumbering of the
        # scheme in the book, used here to take advantage of the fact
        # that Python can use negative indices in lists.
        for l in xrange(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
        return (nabla_b, nabla_w)
 #function for evaluating how the network performed over the test data.
    def evaluate(self, test_data):
            """Return the number of test inputs for which the neural
            network outputs the correct result. Note that the neural
            network's output is assumed to be the index of whichever
            neuron in the final layer has the highest activation."""
            test_results = [(np.argmax(self.feedforward(x)), y)
                            for (x, y) in test_data]
            return sum(int(x == y) for (x, y) in test_results)
# isolating the return vector for the partial derivatives  of the cost function.
    def cost_derivative(self, output_activations, y):
            """Return the vector of partial derivatives \partial C_x /
            \partial a for the output activations."""
            return (output_activations-y)


<>:114: SyntaxWarning: invalid escape sequence '\p'
<>:114: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_1879/2595355911.py:114: SyntaxWarning: invalid escape sequence '\p'
  """Return the vector of partial derivatives \partial C_x /


In [ ]:
#### Miscellaneous functions
def sigmoid(z):
    """The sigmoid function."""
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    """Derivative of the sigmoid function."""
    return sigmoid(z)*(1-sigmoid(z))

In [ ]:
import mnist_loader

In [ ]:
training_data, validation_data, test_data = mnist_loader.load_data_wrapper()

In [ ]:
import network
net = network.Network([784, 30, 10])
net.SGD(training_data, 30, 10, 3.0, test_data=test_data)

Epoch 0: 9052 / 10000, took 8.32 seconds
Epoch 1: 9218 / 10000, took 6.93 seconds
Epoch 2: 9314 / 10000, took 8.30 seconds
Epoch 3: 9321 / 10000, took 7.37 seconds
Epoch 4: 9346 / 10000, took 7.92 seconds
Epoch 5: 9317 / 10000, took 8.37 seconds
Epoch 6: 9419 / 10000, took 6.99 seconds
Epoch 7: 9407 / 10000, took 8.28 seconds
Epoch 8: 9403 / 10000, took 7.00 seconds
Epoch 9: 9445 / 10000, took 8.28 seconds
Epoch 10: 9424 / 10000, took 7.72 seconds
Epoch 11: 9466 / 10000, took 7.38 seconds
Epoch 12: 9431 / 10000, took 8.34 seconds
Epoch 13: 9434 / 10000, took 6.97 seconds
Epoch 14: 9417 / 10000, took 8.45 seconds
Epoch 15: 9455 / 10000, took 7.28 seconds
Epoch 16: 9451 / 10000, took 8.07 seconds
Epoch 17: 9465 / 10000, took 8.26 seconds
Epoch 18: 9474 / 10000, took 7.04 seconds
Epoch 19: 9469 / 10000, took 8.35 seconds
Epoch 20: 9489 / 10000, took 7.00 seconds
Epoch 21: 9487 / 10000, took 8.38 seconds
Epoch 22: 9481 / 10000, took 7.51 seconds
Epoch 23: 9491 / 10000, took 7.55 seconds
Ep

In [ ]:
net = network.Network([784, 30, 10])
net.SGD(training_data, 30, 10, 100.0, test_data=test_data)

Epoch 0: 1135 / 10000, took 7.00 seconds
Epoch 1: 1135 / 10000, took 8.03 seconds
Epoch 2: 1135 / 10000, took 7.94 seconds
Epoch 3: 1135 / 10000, took 7.18 seconds
Epoch 4: 1135 / 10000, took 8.26 seconds
Epoch 5: 1135 / 10000, took 7.10 seconds
Epoch 6: 1135 / 10000, took 8.25 seconds
Epoch 7: 1135 / 10000, took 7.09 seconds
Epoch 8: 1135 / 10000, took 7.93 seconds
Epoch 9: 1135 / 10000, took 8.03 seconds
Epoch 10: 1135 / 10000, took 6.93 seconds
Epoch 11: 1135 / 10000, took 8.16 seconds
Epoch 12: 1135 / 10000, took 6.87 seconds
Epoch 13: 1135 / 10000, took 8.15 seconds
Epoch 14: 1135 / 10000, took 7.11 seconds
Epoch 15: 1135 / 10000, took 7.86 seconds
Epoch 16: 1135 / 10000, took 7.96 seconds
Epoch 17: 1135 / 10000, took 6.88 seconds
Epoch 18: 1135 / 10000, took 8.26 seconds
Epoch 19: 1135 / 10000, took 6.85 seconds
Epoch 20: 1135 / 10000, took 8.26 seconds
Epoch 21: 1135 / 10000, took 7.05 seconds
Epoch 22: 1135 / 10000, took 8.01 seconds
Epoch 23: 1135 / 10000, took 8.01 seconds
Ep

In [ ]:
"""Try creating a network with just two layers
- an input and an output layer,
no hidden layer - with 784 and 10 neurons, respectively.
Train the network using stochastic gradient descent.
What classification accuracy can you achieve? """

net = network.Network([784, 10])
net.SGD(training_data, 30, 10, 1, test_data=test_data)

Epoch 0: 5239 / 10000, took 2.62 seconds
Epoch 1: 5545 / 10000, took 2.59 seconds
Epoch 2: 5651 / 10000, took 4.09 seconds
Epoch 3: 6469 / 10000, took 2.56 seconds
Epoch 4: 6523 / 10000, took 2.60 seconds
Epoch 5: 6519 / 10000, took 2.59 seconds
Epoch 6: 6537 / 10000, took 3.17 seconds
Epoch 7: 6551 / 10000, took 3.42 seconds
Epoch 8: 6552 / 10000, took 2.61 seconds
Epoch 9: 6566 / 10000, took 2.55 seconds
Epoch 10: 6563 / 10000, took 2.70 seconds
Epoch 11: 6567 / 10000, took 3.97 seconds
Epoch 12: 6581 / 10000, took 2.58 seconds
Epoch 13: 6588 / 10000, took 2.57 seconds
Epoch 14: 6595 / 10000, took 2.55 seconds
Epoch 15: 6600 / 10000, took 2.94 seconds
Epoch 16: 6612 / 10000, took 3.62 seconds
Epoch 17: 6619 / 10000, took 2.58 seconds
Epoch 18: 6757 / 10000, took 2.57 seconds
Epoch 19: 7482 / 10000, took 2.60 seconds
Epoch 20: 7497 / 10000, took 3.73 seconds
Epoch 21: 7491 / 10000, took 2.87 seconds
Epoch 22: 7505 / 10000, took 2.57 seconds
Epoch 23: 7511 / 10000, took 2.57 seconds
Ep